# Anveshak: Open-World Remote Sensing Object Detector
### Techfest 2026-27 | IITB-HRTI Centre of Excellence
---
This turnkey notebook handles GPU training, continual few-shot learning, and evaluation directly inside Google Colab with seamless Google Drive & GitHub synchronization.

## 1. Verify GPU & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive
Your persistent datasets and checkpoints will be loaded from and saved to Google Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_ANVESHAK_DIR = '/content/drive/MyDrive/Anveshak'
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/submissions", exist_ok=True)
print("[✓] Google Drive connected successfully.")

## 3. Clone / Update Repository from GitHub

In [ ]:
# Replace with your GitHub Repository URL
GITHUB_REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO_NAME>.git"
REPO_NAME = "IITBBB"

%cd /content
if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO_URL} {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

%cd /content/{REPO_NAME}
!pip install -r requirements.txt -q

## 4. Stage Datasets to Fast Colab Local NVMe SSD
Training directly on mounted Google Drive files causes severe I/O bottlenecks. We extract zip files to local `/content/data` for maximum training speed.

In [ ]:
import zipfile

!mkdir -p /content/data
drive_data_path = f"{DRIVE_ANVESHAK_DIR}/datasets"

for zip_file in os.listdir(drive_data_path):
    if zip_file.endswith(".zip"):
        print(f"Unpacking {zip_file} to /content/data/...")
        with zipfile.ZipFile(os.path.join(drive_data_path, zip_file), 'r') as zip_ref:
            zip_ref.extractall("/content/data/")

print("[✓] All datasets unpacked to local Colab disk.")
!ls -lh /content/data

## 5. Train Detector on Set A (From Scratch - Random Initialization)

In [ ]:
# Execute Set A training script
!python -m src.training.train_set_a \
    --config configs/base_config.yaml \
    --data_dir /content/data \
    --output_dir checkpoints/

# Back up Set A weights to Google Drive immediately
!cp checkpoints/set_a_best.pth {DRIVE_ANVESHAK_DIR}/checkpoints/

## 6. Continual Few-Shot Learning on Set B
Continues the *exact same model lineage* on Set B (10 new classes) with distillation loss and cosine classifier margin to retain Set A classes.

In [ ]:
!python -m src.training.train_set_b \
    --config configs/base_config.yaml \
    --checkpoint_in checkpoints/set_a_best.pth \
    --data_dir /content/data \
    --output_checkpoint checkpoints/round1_final.pth

# Sync final checkpoint to Google Drive
!cp checkpoints/round1_final.pth {DRIVE_ANVESHAK_DIR}/checkpoints/
print("[✓] Round 1 Checkpoint saved and backed up to Google Drive.")

## 7. Inference & Official Conflict Rule Resolution
Generates raw predictions on unlabelled Round 1 test images, applies conflict filtering (IoU >= 0.5 highest score rule), and exports `predictions.csv`.

In [ ]:
!python -m src.inference.predict \
    --checkpoint checkpoints/round1_final.pth \
    --test_images_dir /content/data/test_images \
    --output_csv submission/predictions.csv

# Copy predictions to Google Drive
!cp submission/predictions.csv {DRIVE_ANVESHAK_DIR}/submissions/

## 8. Validate Predictions with Organizer's Script

In [ ]:
# Run validate_predictions.py
!python validate_predictions.py submission/predictions.csv